# Scaffold-tuner Experiments

## Preparation

### Import library

In [1]:
import os, random
import pandas as pd
from tqdm import tqdm

import rdkit
from rdkit import Chem
# from rdkit.Chem import Descriptors, rdMolDescriptors

import sys
sys.path.append("..")

from configs.benchmark_config import N_PARENTS, RANDOM_SEED
from utils.descriptors import calc_descriptors

print(f"Pandas: {pd.__version__}, RDKit: {rdkit.__version__}")

Pandas: 3.0.5, RDKit: 2026.03.5


## Collect compound data from ChEMBL

In [2]:
from chembl_webresource_client.settings import Settings

Settings.Instance().TIMEOUT = 60
Settings.Instance().TOTAL_RETRIES = 10

from chembl_webresource_client.new_client import new_client

/Users/tesak/anaconda3/envs/scaffold-tuner-experiments/lib/python3.11/site-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


In [3]:
SAVE_EVERY = 500

OUTPUT_FILE = "../data/chembl_parent_candidates.csv"

# =========================
# Resume previous progress
# =========================
if os.path.exists(OUTPUT_FILE):
    df_saved = pd.read_csv(OUTPUT_FILE)
    records = df_saved.to_dict("records")
    seen_smiles = set(df_saved["smiles"].dropna())
    print(f"Resuming from {len(records)} saved unique compounds.")
else:
    records = []
    seen_smiles = set()
    print("Starting new collection.")

Resuming from 100000 saved unique compounds.


In [4]:
try:
    for mol in tqdm(results, desc="Collecting ..."):
        if len(records) >= 100 * N_PARENTS:
            break
        
        try:
            smi = mol.get("molecule_structures", {}).get("canonical_smiles")
            cid = mol.get("molecule_chembl_id")
            
            if not smi or not cid:
                continue
    
            rdmol = Chem.MolFromSmiles(smi)
            
            if rdmol is None:
                continue
    
            canonical_smi = Chem.MolToSmiles(rdmol, canonical=True, isomericSmiles=True)
            
            if canonical_smi in seen_smiles:
                continue
    
            seen_smiles.add(canonical_smi)

            desc = calc_descriptors(rdmol)

            # collect chembl data
            records.append({
                "chembl_id": cid,
                "smiles": canonical_smi,
                "mw": Descriptors.MolWt(rdmol),
                "hbd": desc["hbd"],
                "hba": desc["hba"],
                "logp": Descriptors.MolLogP(rdmol),
                "rotb": desc["rotb"],
                "ar": desc["ar"],
            })
            
            # 500件ごとに途中保存
            if len(records) % SAVE_EVERY == 0:
                pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)
                print(f"Saved: {len(records)} compounds")
    
        except Exception:
            continue

except Exception as e:
    print(f"Connection interrupted: {e}")

finally:
    pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)
    print(f"Final saved: {len(records)} compounds")

Connection interrupted: name 'results' is not defined
Final saved: 100000 compounds


### Output randomly selected 1000 parents

In [5]:
df_all = pd.DataFrame(records)
n_before_dedup = len(df_all)
df_all = df_all.drop_duplicates(subset="smiles").reset_index(drop=True)

df = df_all.sample(n=N_PARENTS, random_state=RANDOM_SEED).reset_index(drop=True)
df.to_csv("../data/chembl_1000_parents.csv", index=False)

print(df.shape)
df.head()

(1000, 8)


,chembl_id,smiles,mw,hbd,hba,logp,rotb,ar
0,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,367.449,2,4,4.49360,9,2
1,CHEMBL91137,CCCc1nnc([S+]([O-])Cc2ncc(C)c(OC)c2C)o1,309.391,0,6,2.35034,6,2
2,CHEMBL441229,C=C1C(O)C(O)C(O)C(O)C1O,176.168,5,5,-2.63930,0,0
3,CHEMBL1068,NC(=O)N1c2ccccc2CC(=O)c2ccccc21,252.273,1,2,2.64220,0,2
4,CHEMBL105373,COc1cc(-n2sc3ncccc3c2=O)cc(OC)c1OC,318.354,0,6,2.47300,4,3
